<a href="https://colab.research.google.com/github/MarceloFM1962/agentes-2026-2-equipe-001/blob/main/enc09_rag_Marcelo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Encontro 9 — RAG a serviço do agente

**Tópicos Especiais em IA · Agentes Inteligentes · IFES Serra · 2026/2**

> **A pergunta de hoje.** Quando a resposta não está no modelo nem numa
> consulta estruturada, mas num **documento**, como ela entra na solução, e
> **quem decide ir buscá-la**?

## Parte 0 — Preparo

Duas chaves, e pela primeira vez no semestre elas são de **provedores
diferentes**: Groq para gerar texto, Gemini para transformar texto em vetor.

A `GEMINI_API_KEY` é **opcional**: sem ela a aula roda inteira com o
recuperador por palavra, e só a coluna de comparação fica vazia.

In [1]:
# ---------------------------------------------------------------------
# PREPARO. Versões ajustadas para evitar incompatibilidade com Python 3.13
# ---------------------------------------------------------------------
%pip install -q "openai==1.109.1" "scikit-learn>=1.6.0" "numpy>=2.1.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 6.3 MB/s eta 0:00:00


In [3]:
import os, sys, json, time, types, unicodedata, subprocess
import numpy as np
from openai import OpenAI


def obter_chave(nome: str, obrigatoria: bool = True) -> str:
    """Le um segredo dos Secrets do Colab; fora do Colab, da variavel de ambiente."""
    try:
        from google.colab import userdata
        try:
            return userdata.get(nome)
        except Exception:
            valor = None
    except ImportError:
        valor = os.getenv(nome)
    if not valor:
        valor = os.getenv(nome)
    if not valor and obrigatoria:
        raise RuntimeError(f"Defina {nome} nos Secrets do Colab ou no ambiente.")
    return valor or ""


# --- GERACAO: Groq, como em toda a disciplina -------------------------
LLM_MODEL = "openai/gpt-oss-20b"
cliente = OpenAI(base_url="https://api.groq.com/openai/v1",
                 api_key=obter_chave("GROQ_API_KEY"))

# --- EMBEDDINGS: Gemini, decisao 7 do AGENTS.md -----------------------
# O Gemini entra SO aqui, e so nesta aula. Ele nao serve para ferramentas
# (erro 400 de thought_signature, verificado em 28/07/2026), mas serve
# muito bem para transformar texto em vetor.
GEMINI_API_KEY = obter_chave("GEMINI_API_KEY", obrigatoria=False)
EMB_MODEL = "gemini-embedding-001"
cliente_emb = None
if GEMINI_API_KEY:
    cliente_emb = OpenAI(
        base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
        api_key=GEMINI_API_KEY)

PRECOS = {"openai/gpt-oss-20b":  {"entrada": 0.075, "saida": 0.30},
          "openai/gpt-oss-120b": {"entrada": 0.150, "saida": 0.60}}
TPM = 8_000

print("geracao :", LLM_MODEL, "(Groq)")
print("vetores :", EMB_MODEL, "(Gemini)" if cliente_emb else "SEM CHAVE -- ver a celula do plano B")
print()

geracao : openai/gpt-oss-20b (Groq)
vetores : gemini-embedding-001 (Gemini)



In [5]:
def _espera_sugerida(erro, padrao: float) -> float:
    try:
        cab = getattr(getattr(erro, "response", None), "headers", {}) or {}
        v = cab.get("retry-after") or cab.get("Retry-After")
        if v:
            return float(v)
    except (TypeError, ValueError):
        pass
    return padrao


def chamar(mensagens, temperatura: float = 0.0, tentativas: int = 5,
           max_tokens: int = 900, **extra):
    """Chamada tolerante a 429/503, igual a dos Encontros 4 a 7."""
    espera = 2.0
    for t in range(tentativas):
        try:
            r = cliente.chat.completions.create(
                model=LLM_MODEL, messages=mensagens,
                temperature=temperatura, max_tokens=max_tokens, **extra)
            if r.choices[0].finish_reason == "length":
                print("  [AVISO: resposta CORTADA por max_tokens.]")
            return r
        except Exception as e:
            codigo = getattr(e, "status_code", None)
            if codigo not in (429, 500, 502, 503, 504) or t == tentativas - 1:
                raise
            pausa = _espera_sugerida(e, espera)
            print(f"  [{codigo}] esperando {pausa:.0f}s  (nao invalida a medicao)")
            time.sleep(pausa)
            espera *= 2
    raise RuntimeError("todas as tentativas falharam")


def custo_usd(entrada: int, saida: int) -> float:
    p = PRECOS[LLM_MODEL]
    return (entrada * p["entrada"] + saida * p["saida"]) / 1_000_000


print("chamada e custo prontos.")

chamada e custo prontos.


## Parte 1 — O corpus, e por que ele é inventado

Oito documentos da **Vértice Instrumentos**, fabricante de um medidor de
pressão que não existe. Manual, políticas, boletim de serviço, FAQ, glossário.

O corpus é sintético pela regra do ADR-001, e aqui isso rende um segundo
benefício: **com manual inventado dá para saber a resposta certa**. Com manual
real você mede impressão.

In [6]:
# ---------------------------------------------------------------------
# O CORPUS. Oito documentos de uma empresa que nao existe: a Vertice
# Instrumentos, fabricante do medidor de pressao VX-200.
#
# Tudo sintetico, como manda o ADR-001 -- e aqui o sintetico tem uma
# segunda funcao: PERMITE SABER A RESPOSTA CERTA. Com manual real, voce
# mede impressao. Com manual inventado, voce mede acerto.
# ---------------------------------------------------------------------

DOCS = {
 "D-01": {"titulo": "Manual VX-200, secao 3: instalacao", "texto":
  "O medidor VX-200 deve ser instalado em superficie plana, longe de fontes de "
  "vibracao. O aterramento do chassi e obrigatorio e deve usar cabo de 4 mm2 "
  "ligado ao barramento de terra do painel. Apos a instalacao o tecnico preenche "
  "o LTI e envia copia para a assistencia. Sem LTI registrado a cobertura "
  "estendida nao se aplica."},

 "D-02": {"titulo": "Politica de garantia da linha VX", "texto":
  "Todo medidor da linha VX tem garantia de 36 meses contados da data da nota "
  "fiscal. A garantia cobre defeito de fabricacao e falha de componente. Nao "
  "cobre dano por surto eletrico, imersao ou uso fora da faixa especificada. A "
  "cobertura estendida de 60 meses exige LTI registrado e calibracao anual em "
  "assistencia autorizada."},

 "D-03": {"titulo": "Politica de troca e devolucao", "texto":
  "Equipamento sem uso pode ser devolvido em ate 45 dias da entrega, na "
  "embalagem original. Equipamento ja instalado nao e devolvido: entra em "
  "avaliacao tecnica. O frete de devolucao e por conta da Vertice quando o laudo "
  "aponta defeito de fabricacao."},

 "D-04": {"titulo": "FAQ: o erro E-12", "texto":
  "O erro E-12 aparece quando o sensor de pressao nao responde em tres leituras "
  "seguidas. Na maioria dos casos o cabo do sensor esta frouxo: reapertar o "
  "conector e reiniciar o medidor resolve. Se o E-12 voltar em menos de 24 horas, "
  "o sensor precisa de troca."},

 "D-05": {"titulo": "Boletim de servico BS-2026-03", "texto":
  "Este boletim SUBSTITUI o procedimento de aterramento da secao 3 do manual, a "
  "partir de marco de 2026. O cabo de 4 mm2 deixa de ser aceito em instalacoes "
  "que tenham inversor de frequencia no mesmo painel. Nessas instalacoes use cabo "
  "de 10 mm2 e barra de equipotencializacao dedicada. Instalacoes anteriores a "
  "marco de 2026 devem ser revisadas na proxima calibracao."},

 "D-06": {"titulo": "Especificacoes tecnicas do VX-200", "texto":
  "Faixa de medicao: 0 a 25 bar. Precisao: 0,1 por cento do fundo de escala. "
  "Temperatura de operacao: -10 a 60 graus Celsius. Alimentacao: 24 V em corrente "
  "continua. Saidas: 4 a 20 mA e Modbus RTU. O VX-200 mede pressao apenas: nao "
  "mede vazao nem temperatura de processo."},

 "D-07": {"titulo": "Procedimento de calibracao PC-07", "texto":
  "Este procedimento se aplica a todos os medidores da linha VX em uso "
  "industrial. Materiais necessarios: bomba de teste calibrada, manometro padrao "
  "com certificado vigente, chave de 8 mm e formulario FC-02. "
  "Passo 1: isole o medidor da linha de processo e alivie a pressao residual. "
  "Passo 2: conecte o manometro padrao na porta de teste e energize o medidor. "
  "Passo 3: antes de ajustar o zero, aguarde 20 minutos com o medidor energizado. "
  "A deriva termica do sensor so estabiliza depois desse tempo, e ajustar antes e "
  "a causa mais comum de calibracao reprovada. "
  "Passo 4: aplique 25 por cento, 50 por cento, 75 por cento e 100 por cento do "
  "fundo de escala, anotando a leitura em cada ponto. "
  "Passo 5: calcule o erro em cada ponto e registre no formulario FC-02. "
  "Observacao: se o erro passar de 0,2 por cento do fundo de escala em qualquer "
  "ponto, o medidor vai para a assistencia e nao e liberado para uso."},

 "D-08": {"titulo": "Glossario interno", "texto":
  "LTI: Laudo Tecnico de Instalacao, documento preenchido pelo tecnico depois da "
  "instalacao. RMA: numero de autorizacao de retorno de material. FE: fundo de "
  "escala. BS: boletim de servico, documento que altera ou substitui trecho de "
  "manual."},
}

print(f"{len(DOCS)} documentos.")
for _i, _d in DOCS.items():
    print(f"  {_i}  {_d['titulo']:<42} {len(_d['texto']):>4} caracteres")

8 documentos.
  D-01  Manual VX-200, secao 3: instalacao          323 caracteres
  D-02  Politica de garantia da linha VX            323 caracteres
  D-03  Politica de troca e devolucao               247 caracteres
  D-04  FAQ: o erro E-12                            255 caracteres
  D-05  Boletim de servico BS-2026-03               364 caracteres
  D-06  Especificacoes tecnicas do VX-200           268 caracteres
  D-07  Procedimento de calibracao PC-07            899 caracteres
  D-08  Glossario interno                           237 caracteres


### Cinco perguntas, cinco modos de falha

Cada pergunta foi escrita para quebrar o RAG de um jeito diferente. Repare na
**verdade de terra em duas colunas**: qual documento tinha de ser recuperado, e
o que uma resposta certa contém. São coisas distintas, e separá-las é o que
permite consertar a coisa certa depois.

In [7]:
# ---------------------------------------------------------------------
# AS PERGUNTAS, e a VERDADE DE TERRA de cada uma. Duas colunas de
# verdade, e a distincao entre elas e a aula inteira:
#
#   docs   -> QUAL documento tem de ser recuperado
#   termos -> o que uma resposta certa quase obrigatoriamente contem
#
# Recuperar certo e responder certo sao coisas diferentes, e medi-las
# separado e o que permite consertar a coisa certa.
# ---------------------------------------------------------------------

PERGUNTAS = [
 {"id": "Q-01", "texto": "Por quanto tempo o VX-200 tem garantia?"},
 {"id": "Q-02", "texto": "O que e o LTI e quem preenche?"},
 {"id": "Q-03", "texto": "O que fazer antes de ajustar o zero na calibracao?"},
 {"id": "Q-04", "texto": "Que bitola de cabo devo usar no aterramento do VX-200?"},
 #{"id": "Q-04", "texto": "o aparelho veio com problema de fábrica, por quanto tempo posso acionar o fabricante?"},
 {"id": "Q-05", "texto": "Qual o consumo do VX-200 em modo de espera?"},
]

VERDADE = {
 "Q-01": {"docs": {"D-02"}, "termos": ["36"],
          "nota": "CONTRA O MUNDO: o padrao de mercado seria 12 meses. Se o "
                  "modelo responder 12, a resposta nao veio do corpus."},
 "Q-02": {"docs": {"D-08", "D-01"}, "termos": ["laudo"],
          "nota": "SIGLA: 'LTI' nao tem significado semantico. E o caso em que "
                  "a busca por palavra ganha da busca por vetor."},
 "Q-03": {"docs": {"D-07"}, "termos": ["20 minuto", "deriva"],
          "nota": "TRECHO FUNDO: a frase decisiva esta no meio de um documento "
                  "longo. Se o pedaco for pequeno demais, ela perde o assunto."},
 "Q-04": {"docs": {"D-05"}, "termos": ["10 mm"],
          "nota": "ARMADILHA DE VERSAO: o D-01 responde '4 mm2' com toda a "
                  "confianca, e esta DESATUALIZADO desde marco de 2026. O "
                  "D-05 e quem corrige, e ele e MENOS parecido com a "
                  "pergunta. Com k=1 a armadilha fecha."},
 "Q-05": {"docs": set(), "termos": ["nao encontr", "nao consta", "nao ha",
                                    "nao foi possivel", "nao esta"],
          "nota": "SEM RESPOSTA NO CORPUS. A resposta certa e dizer que nao "
                  "encontrou. Inventar aqui e o pior resultado da aula."},
}

print(f"{len(PERGUNTAS)} perguntas, cada uma com um modo de falha diferente:")
for _q in PERGUNTAS:
    print(f"  {_q['id']}  {VERDADE[_q['id']]['nota'].split(':')[0]}")

5 perguntas, cada uma com um modo de falha diferente:
  Q-01  CONTRA O MUNDO
  Q-02  SIGLA
  Q-03  TRECHO FUNDO
  Q-04  ARMADILHA DE VERSAO
  Q-05  SEM RESPOSTA NO CORPUS. A resposta certa e dizer que nao encontrou. Inventar aqui e o pior resultado da aula.


## Parte 2 — As quatro decisões de um RAG

Todo RAG, por mais sofisticado, é feito destas quatro escolhas:

| | Decisão | Onde erra |
|---|---|---|
| 1 | **o tamanho do pedaço** | pequeno perde o assunto, grande dilui o trecho |
| 2 | **como achar o pedaço** | por palavra, por significado, ou os dois |
| 3 | **o que vai no *prompt*** | quantos pedaços, em que ordem, com que instrução |
| 4 | **como saber se acertou** | e esta é a que quase todo projeto pula |

Nenhuma delas tem valor certo. Todas têm valor **medido**.

In [8]:
# ---------------------------------------------------------------------
# DECISAO 1 de 4: O TAMANHO DO PEDACO.
# Nenhuma chamada de rede nesta celula.
# ---------------------------------------------------------------------

def partir(texto: str, tam: int = 400, sobreposicao: int = 80) -> list:
    """Corta o texto em pedacos de `tam` caracteres, com sobreposicao.

    A sobreposicao existe para a frase que cai em cima do corte nao
    desaparecer das duas metades.
    """
    if tam <= sobreposicao:
        raise ValueError("a sobreposicao tem de ser menor que o pedaco")
    pedacos, i = [], 0
    while i < len(texto):
        pedacos.append(texto[i:i + tam].strip())
        i += tam - sobreposicao
    return [p for p in pedacos if p]


def indexar(tam: int = 400, sobreposicao: int = 80) -> list:
    """Monta a lista de pedacos de todo o corpus, cada um sabendo sua origem."""
    base = []
    for doc_id, d in DOCS.items():
        for k, p in enumerate(partir(d["texto"], tam, sobreposicao)):
            base.append({"id": f"{doc_id}#{k}", "doc": doc_id,
                         "titulo": d["titulo"], "texto": p})
    return base


BASE = indexar()
print(f"{len(BASE)} pedacos com tam=400, sobreposicao=80.")
print()
print("O mesmo corpus com tres tamanhos:")
for _t in (150, 400, 900):
    print(f"  tam={_t:>3}  ->  {len(indexar(_t, min(80, _t // 4))):>3} pedacos")
print()
print("Pedaco pequeno acha o trecho e perde o contexto. Pedaco grande traz o")
print("contexto e dilui o trecho. Nao ha valor certo: ha valor MEDIDO.")

13 pedacos com tam=400, sobreposicao=80.

O mesmo corpus com tres tamanhos:
  tam=150  ->   30 pedacos
  tam=400  ->   13 pedacos
  tam=900  ->    9 pedacos

Pedaco pequeno acha o trecho e perde o contexto. Pedaco grande traz o
contexto e dilui o trecho. Nao ha valor certo: ha valor MEDIDO.


### Decisão 2, primeira forma: achar por palavra

TF-IDF, que é contagem de palavra com peso. Sem chave, sem rede, sem custo.
É a **linha de base**: a forma barata vem
primeiro, para a cara ter contra o que ser comparada.

In [9]:
# ---------------------------------------------------------------------
# RECUPERADOR 1: POR PALAVRA (TF-IDF). Sem rede, sem chave, sem custo.
# E a linha de base -- a mesma logica do Encontro 7: a forma barata
# primeiro, para o caro ter contra o que ser comparado.
# ---------------------------------------------------------------------
from sklearn.feature_extraction.text import TfidfVectorizer

_vetorizador = TfidfVectorizer(lowercase=True, ngram_range=(1, 2))
_M_lex = _vetorizador.fit_transform([p["texto"] for p in BASE])


def cosseno(a, b):
    """Cosseno entre um vetor e uma matriz de vetores."""
    a = np.asarray(a, dtype=float).ravel()
    b = np.asarray(b, dtype=float)
    na = np.linalg.norm(a) or 1.0
    nb = np.linalg.norm(b, axis=1)
    nb[nb == 0] = 1.0
    return (b @ a) / (nb * na)


def buscar_lexical(pergunta: str, k: int = 3) -> list:
    sim = cosseno(_vetorizador.transform([pergunta]).toarray(), _M_lex.toarray())
    ordem = np.argsort(-sim)[:k]
    return [dict(BASE[i], score=float(sim[i])) for i in ordem]


print("recuperador lexical pronto sobre", _M_lex.shape[1], "termos.")
for _r in buscar_lexical("O que e o LTI?", k=3):
    print(f"  {_r['score']:.3f}  {_r['id']}  {_r['texto'][:60]}...")

recuperador lexical pronto sobre 635 termos.
  0.136  D-08#0  LTI: Laudo Tecnico de Instalacao, documento preenchido pelo ...
  0.111  D-01#0  O medidor VX-200 deve ser instalado em superficie plana, lon...
  0.059  D-05#0  Este boletim SUBSTITUI o procedimento de aterramento da seca...


### Decisão 2, segunda forma: achar por significado

Aqui entra o Gemini, e **só aqui**. A decisão 7 do `AGENTS.md` o restringe a
*embeddings* porque ele falha em *tool calling* no canal compatível com a
OpenAI. Transformar texto em vetor ele faz muito bem.

In [10]:
# ---------------------------------------------------------------------
# RECUPERADOR 2: POR SIGNIFICADO (vetores do Gemini).
# Esta celula fala com a rede UMA vez, e so uma: os vetores do corpus
# sao calculados de uma vez e guardados em memoria.
# ---------------------------------------------------------------------
# REDE

def vetorizar(textos: list) -> np.ndarray:
    """Texto -> vetor, pelo endpoint compativel com OpenAI do Gemini."""
    r = cliente_emb.embeddings.create(model=EMB_MODEL, input=textos)
    return np.array([d.embedding for d in r.data], dtype=float)


SEMANTICA_OK = False
if cliente_emb is not None:
    try:
        _M_sem = vetorizar([p["texto"] for p in BASE])
        SEMANTICA_OK = True
        print("vetores do corpus:", _M_sem.shape, "(pedacos x dimensoes)")
    except Exception as e:
        print("[FALHOU o Gemini]", type(e).__name__, str(e)[:160])
        print("Siga com o recuperador lexical; a aula continua inteira.")
else:
    print("Sem GEMINI_API_KEY. Esta aula roda sem ela, mas a comparacao")
    print("lexical x semantica fica so com uma coluna.")


def buscar_semantica(pergunta: str, k: int = 3) -> list:
    if not SEMANTICA_OK:
        return []
    sim = cosseno(vetorizar([pergunta])[0], _M_sem)
    ordem = np.argsort(-sim)[:k]
    return [dict(BASE[i], score=float(sim[i])) for i in ordem]

vetores do corpus: (13, 3072) (pedacos x dimensoes)


In [12]:
# REDE (depende da celula anterior). O VETOR EXISTE E DA PARA VER.
_frases = ["o medidor esta com defeito de fabricacao",
           "o aparelho veio com problema de fabrica",
           "o prazo de garantia e de tres anos",
           "a bomba de vacuo precisa de oleo",
           "LTI"]

if SEMANTICA_OK:
    _V = vetorizar(_frases)
    print("dimensao de cada vetor:", _V.shape[1])
    print()
    print(" " * 34, "  ".join(f"{i+1:>5}" for i in range(len(_frases))))
    for _i, _f in enumerate(_frases):
        _linha = "  ".join(f"{v:5.2f}" for v in cosseno(_V[_i], _V))
        print(f"{_i+1}. {_f[:30]:<31} {_linha}")
    print()
    print("As frases 1 e 2 nao tem quase nenhuma palavra em comum e ficam perto.")
    print("E isso que a busca por palavra nao consegue fazer.")
    print()
    print("Agora olhe a linha 5, que e so uma SIGLA, e responda voce: ela esta")
    print("perto de quem? HIPOTESE a testar na Parte 3, nao conclusao: sigla")
    print("carrega pouco significado, entao a busca por vetor pode se sair")
    print("PIOR que a busca por palavra justamente nela.")
else:
    print("(sem chave do Gemini: esta demonstracao precisa dos vetores)")

dimensao de cada vetor: 3072

                                       1      2      3      4      5
1. o medidor esta com defeito de    1.00   0.82   0.65   0.56   0.52
2. o aparelho veio com problema d   0.82   1.00   0.67   0.57   0.52
3. o prazo de garantia e de tres    0.65   0.67   1.00   0.60   0.54
4. a bomba de vacuo precisa de ol   0.56   0.57   0.60   1.00   0.48
5. LTI                              0.52   0.52   0.54   0.48   1.00

As frases 1 e 2 nao tem quase nenhuma palavra em comum e ficam perto.
E isso que a busca por palavra nao consegue fazer.

Agora olhe a linha 5, que e so uma SIGLA, e responda voce: ela esta
perto de quem? HIPOTESE a testar na Parte 3, nao conclusao: sigla
carrega pouco significado, entao a busca por vetor pode se sair
PIOR que a busca por palavra justamente nela.


## Parte 3 — A medida que não custa nada

**Qualidade de recuperação se mede sem chamar o modelo de linguagem.** Essa é
a melhor notícia da aula, e a mais ignorada em projeto: dá para rodar centenas
de perguntas, variar pedaço e *k*, e comparar recuperadores **de graça**.

O que se mede aqui é só uma coisa: o documento certo entrou entre os *k*
primeiros?

In [13]:
# ---------------------------------------------------------------------
# A MEDIDA QUE NAO CUSTA NADA: o pedaco certo entrou nos k primeiros?
# Nenhuma chamada ao modelo de linguagem nesta celula. Qualidade de
# recuperacao se mede de GRACA, e isso e a melhor noticia da aula.
# ---------------------------------------------------------------------

def acertou_em_k(trechos: list, pergunta_id: str) -> bool:
    alvo = VERDADE[pergunta_id]["docs"]
    if not alvo:                       # Q-05 nao tem documento certo
        return True                    # nao recuperar nada errado ja basta
    return bool(alvo & {t["doc"] for t in trechos})


def tabela_acerto(ks=(1, 3, 5)) -> None:
    cabec = ["pergunta"] + [f"lex@{k}" for k in ks]
    if SEMANTICA_OK:
        cabec += [f"sem@{k}" for k in ks]
    print(" ".join(f"{c:>9}" for c in cabec))
    print("-" * (10 * len(cabec)))
    soma = {c: 0 for c in cabec[1:]}
    for q in PERGUNTAS:
        linha = [q["id"]]
        for k in ks:
            ok = acertou_em_k(buscar_lexical(q["texto"], k), q["id"])
            linha.append("SIM" if ok else "nao")
            soma[f"lex@{k}"] += ok
        if SEMANTICA_OK:
            for k in ks:
                ok = acertou_em_k(buscar_semantica(q["texto"], k), q["id"])
                linha.append("SIM" if ok else "nao")
                soma[f"sem@{k}"] += ok
        print(" ".join(f"{c:>9}" for c in linha))
    print("-" * (10 * len(cabec)))
    print(" ".join([f"{'total':>9}"] +
                   [f"{soma[c]}/{len(PERGUNTAS):<7}" for c in cabec[1:]]))


NOMES_ESPERADOS = ["DOCS", "PERGUNTAS", "VERDADE", "partir", "indexar",
                   "buscar_lexical", "acertou_em_k", "responder", "quadrante"]
print("medida de recuperacao pronta. Zero tokens gastos ate aqui.")

medida de recuperacao pronta. Zero tokens gastos ate aqui.


In [14]:
# REDE (depende dos dois recuperadores). A TABELA QUE NAO CUSTA TOKEN.
tabela_acerto()
print()
print("Leia a coluna, nao a celula: onde o lexical ganha e onde ele perde.")

 pergunta     lex@1     lex@3     lex@5     sem@1     sem@3     sem@5
----------------------------------------------------------------------
     Q-01       SIM       SIM       SIM       SIM       SIM       SIM
     Q-02       SIM       SIM       SIM       SIM       SIM       SIM
     Q-03       SIM       SIM       SIM       SIM       SIM       SIM
     Q-04       nao       SIM       SIM       nao       SIM       SIM
     Q-05       SIM       SIM       SIM       SIM       SIM       SIM
----------------------------------------------------------------------
    total 4/5       5/5       5/5       4/5       5/5       5/5      

Leia a coluna, nao a celula: onde o lexical ganha e onde ele perde.


## Parte 4 — Decisão 3: o que vai no *prompt*

A instrução faz duas coisas:

- **manda citar o documento** em cada afirmação, o que torna o erro visível;
- **autoriza dizer que não achou**, o que dá ao modelo uma saída honesta.

Sem a segunda, a única saída dele quando o trecho não serve é inventar.

In [15]:
# ---------------------------------------------------------------------
# DECISAO 3 de 4: O QUE VAI NO PROMPT.
# A instrucao faz duas coisas, e as duas sao poka-yoke do Encontro 6:
# manda citar o documento, e autoriza dizer que nao achou.
# ---------------------------------------------------------------------

INSTRUCAO_RAG = """Voce responde duvidas sobre os produtos da Vertice Instrumentos
usando SOMENTE os trechos de documentacao fornecidos abaixo.

Responda em ate quatro linhas. Toda afirmacao tem de vir acompanhada do
identificador do documento de onde ela saiu, no formato [D-NN].

Se os trechos fornecidos nao contiverem a resposta, diga exatamente: "Nao
encontrei essa informacao na documentacao." Nao complete com conhecimento
proprio: neste contexto, responder de memoria e um erro mais grave do que
nao responder."""


def montar_contexto(trechos: list) -> str:
    return "\n\n".join(f"[{t['doc']}] {t['titulo']}\n{t['texto']}"
                        for t in trechos)


def responder(pergunta: str, trechos: list) -> dict:
    """Uma chamada ao modelo, com os trechos recuperados no prompt."""
    msgs = [{"role": "system", "content": INSTRUCAO_RAG},
            {"role": "user", "content":
             f"Trechos:\n{montar_contexto(trechos)}\n\nPergunta: {pergunta}"}]
    r = chamar(msgs)
    return {"texto": r.choices[0].message.content or "",
            "entrada": r.usage.prompt_tokens,
            "saida": r.usage.completion_tokens}


def normalizar(t: str) -> str:
    t = unicodedata.normalize("NFD", (t or "").lower())
    return "".join(c for c in t if unicodedata.category(c) != "Mn")


def respondeu_certo(resposta: str, pergunta_id: str) -> bool:
    alvo = VERDADE[pergunta_id]["termos"]
    texto = normalizar(resposta)
    return any(normalizar(a) in texto for a in alvo)


def quadrante(recuperou: bool, acertou: bool) -> str:
    """Os quatro quadrantes, e cada um pede um conserto diferente."""
    if recuperou and acertou:
        return "OK"
    if recuperou and not acertou:
        return "IGNOROU A EVIDENCIA   -> instrucao, nao recuperacao"
    if not recuperou and acertou:
        return "ACERTOU SEM RECUPERAR -> o RAG nao esta dirigindo a resposta"
    return "FALHA COMPLETA        -> recuperacao, nao instrucao"


# --- AUTOTESTE do avaliador, sem rede --------------------------------
assert respondeu_certo("A garantia e de 36 meses [D-02].", "Q-01")
assert not respondeu_certo("A garantia e de 12 meses.", "Q-01")
assert respondeu_certo("Nao encontrei essa informacao na documentacao.", "Q-05")
assert quadrante(False, True).startswith("ACERTOU SEM RECUPERAR")
print("avaliador conferido. Os quatro quadrantes:")
for _r in (True, False):
    for _a in (True, False):
        print(f"  recuperou={str(_r):<5} acertou={str(_a):<5} -> {quadrante(_r, _a)}")

avaliador conferido. Os quatro quadrantes:
  recuperou=True  acertou=True  -> OK
  recuperou=True  acertou=False -> IGNOROU A EVIDENCIA   -> instrucao, nao recuperacao
  recuperou=False acertou=True  -> ACERTOU SEM RECUPERAR -> o RAG nao esta dirigindo a resposta
  recuperou=False acertou=False -> FALHA COMPLETA        -> recuperacao, nao instrucao


### Os quatro quadrantes, que são a medição desta aula

|  | respondeu certo | respondeu errado |
|---|---|---|
| **recuperou o trecho certo** | ok | **ignorou a evidência** · conserte a instrução |
| **não recuperou** | **acertou sem recuperar** · o RAG não está dirigindo nada | falha completa · conserte a recuperação |

O quadrante de baixo à esquerda é o perigoso: parece que funcionou, e não
funcionou. Ele só aparece se alguma pergunta do seu conjunto tiver resposta
**diferente do que o modelo diria sozinho** — e é por isso que a garantia do
VX-200 é de 36 meses e não de 12.

In [16]:
# REDE. As cinco perguntas, recuperando com o lexical, k=3.
# Cada chamada aqui leva uns 900 tokens de entrada. Cinco chamadas em
# sequencia ficam em torno de 5.000, abaixo dos 8.000 por minuto -- por
# isso a espera aqui e de 20s e nao dos 60s do Encontro 7.

RESULTADOS = []
for _i, _q in enumerate(PERGUNTAS):
    _tr = buscar_lexical(_q["texto"], k=3)
    _rec = acertou_em_k(_tr, _q["id"])
    _out = responder(_q["texto"], _tr)
    _cer = respondeu_certo(_out["texto"], _q["id"])
    _out.update(id=_q["id"], recuperou=_rec, certo=_cer,
                quad=quadrante(_rec, _cer),
                custo=custo_usd(_out["entrada"], _out["saida"]),
                docs=[t["doc"] for t in _tr])
    RESULTADOS.append(_out)
    print("=" * 66)
    print(f"{_q['id']}  {_q['texto']}")
    print(f"  recuperou: {_out['docs']}   (esperado: {VERDADE[_q['id']]['docs'] or 'nenhum'})")
    print(f"  resposta : {_out['texto'].strip()[:260]}")
    print(f"  veredito : {_out['quad']}")
    if _i < len(PERGUNTAS) - 1:
        time.sleep(20)

Q-01  Por quanto tempo o VX-200 tem garantia?
  recuperou: ['D-02', 'D-06', 'D-07']   (esperado: {'D-02'})
  resposta : O VX‑200 tem garantia de 36 meses contados da data da nota fiscal. [D‑02]
  veredito : OK
Q-02  O que e o LTI e quem preenche?
  recuperou: ['D-01', 'D-08', 'D-05']   (esperado: {'D-08', 'D-01'})
  resposta : O LTI é o **Laudo Técnico de Instalação**, documento que registra a instalação realizada.  
Ele é **preenchido pelo técnico** imediatamente após concluir a montagem. [D‑08]
  veredito : OK
Q-03  O que fazer antes de ajustar o zero na calibracao?
  recuperou: ['D-07', 'D-07', 'D-05']   (esperado: {'D-07'})
  resposta : Antes de ajustar o zero, aguarde 20 minutos com o medidor energizado para que a deriva térmica do sensor se estabilize. [D-07]
  veredito : OK
Q-04  Que bitola de cabo devo usar no aterramento do VX-200?
  recuperou: ['D-01', 'D-05', 'D-06']   (esperado: {'D-05'})
  resposta : Para instalações sem inversor de frequência, use cabo de 4 mm² (D‑01).  


In [17]:
# REDE (depende da celula anterior). O GRID DO ENCONTRO 9.
print(f"{'pergunta':>9} {'recuperou':>10} {'certo':>6} {'US$':>9}  quadrante")
print("-" * 78)
for _r in RESULTADOS:
    print(f"{_r['id']:>9} {str(_r['recuperou']):>10} {str(_r['certo']):>6} "
          f"{_r['custo']:>9.6f}  {_r['quad']}")
print("-" * 78)
print(f"  recuperacao: {sum(r['recuperou'] for r in RESULTADOS)}/{len(RESULTADOS)}"
      f" | resposta: {sum(r['certo'] for r in RESULTADOS)}/{len(RESULTADOS)}"
      f" | US$ {sum(r['custo'] for r in RESULTADOS):.6f}")
print()
print("A pergunta que decide o conserto NAO e 'o RAG funcionou'.")
print("E: EM QUAL DOS QUATRO QUADRANTES cada caso caiu? Porque consertar")
print("recuperacao e consertar instrucao sao trabalhos diferentes, e quem")
print("mede so 'certo/errado' nao sabe qual dos dois comecar.")

 pergunta  recuperou  certo       US$  quadrante
------------------------------------------------------------------------------
     Q-01       True   True  0.000078  OK
     Q-02       True   True  0.000104  OK
     Q-03       True   True  0.000104  OK
     Q-04       True  False  0.000116  IGNOROU A EVIDENCIA   -> instrucao, nao recuperacao
     Q-05       True   True  0.000062  OK
------------------------------------------------------------------------------
  recuperacao: 5/5 | resposta: 4/5 | US$ 0.000464

A pergunta que decide o conserto NAO e 'o RAG funcionou'.
E: EM QUAL DOS QUATRO QUADRANTES cada caso caiu? Porque consertar
recuperacao e consertar instrucao sao trabalhos diferentes, e quem
mede so 'certo/errado' nao sabe qual dos dois comecar.


In [18]:
# REDE. O quadrante mais perigoso, em duas chamadas.
_q = PERGUNTAS[0]["texto"]

_com = responder(_q, buscar_lexical(_q, k=3))
time.sleep(20)

# Teto MAIOR aqui, de proposito: sem trecho o raciocinio se alonga, e com
# 900 o modelo gastou o orcamento inteiro pensando e devolveu vazio.
_r = chamar([{"role": "system", "content": "Responda em uma linha."},
             {"role": "user", "content": _q}], max_tokens=2000)
_sem = {"texto": _r.choices[0].message.content or "(vazio)",
        "saida": _r.usage.completion_tokens}

print("PERGUNTA:", _q, "\n")
print("COM os trechos :", _com["texto"].strip()[:240])
print(f"   tokens de saida: {_com['saida']}\n")
print("SEM trecho algum:", _sem["texto"].strip()[:240])
print(f"   tokens de saida: {_sem['saida']}\n")
print("Compare as duas contas: SEM o trecho o modelo gasta MAIS saida para")
print("responder PIOR. O contexto nao so melhora a resposta, ele ENCURTA o")
print("raciocinio -- e raciocinio e cobrado como saida, a quatro vezes o")
print("preco da entrada.")

PERGUNTA: Por quanto tempo o VX-200 tem garantia? 

COM os trechos : O VX‑200 tem garantia de 36 meses contados da data da nota fiscal. [D‑02]
   tokens de saida: 129

SEM trecho algum: A garantia do VX‑200 é de 12 meses a partir da data de compra.
   tokens de saida: 136

Compare as duas contas: SEM o trecho o modelo gasta MAIS saida para
responder PIOR. O contexto nao so melhora a resposta, ele ENCURTA o
raciocinio -- e raciocinio e cobrado como saida, a quatro vezes o
preco da entrada.


## Parte 5 — Quebrar de propósito

O método do Encontro 6, e aqui ele sai de graça: **os três experimentos abaixo
não chamam o modelo nenhuma vez.**

In [19]:
# ---------------------------------------------------------------------
# QUEBRAR DE PROPOSITO, e de graca: tudo abaixo sem chamada ao modelo.
# ---------------------------------------------------------------------
_global_base = BASE

print("### 1. O TAMANHO DO PEDACO muda quem e recuperado (Q-03)")
for _tam in (120, 400, 1200):
    BASE = indexar(_tam, min(80, _tam // 4))
    _vz = TfidfVectorizer(lowercase=True, ngram_range=(1, 2))
    _Mz = _vz.fit_transform([p["texto"] for p in BASE])
    _sim = cosseno(_vz.transform([PERGUNTAS[2]["texto"]]).toarray(), _Mz.toarray())
    _top = BASE[int(np.argmax(_sim))]
    _tem = "20 minutos" in _top["texto"]
    print(f"  tam={_tam:>4}  melhor={_top['id']:<8} contem a frase decisiva: "
          f"{'SIM' if _tem else 'NAO'}")
BASE = _global_base
_vetorizador = TfidfVectorizer(lowercase=True, ngram_range=(1, 2))
_M_lex = _vetorizador.fit_transform([p["texto"] for p in BASE])

print()
print("### 2. O k decide se a armadilha de versao fecha (Q-04)")
print(f"    pergunta: {PERGUNTAS[3]['texto']}")
for _k in (1, 3, 5):
    _tr = buscar_lexical(PERGUNTAS[3]["texto"], k=_k)
    _chars = len(montar_contexto(_tr))
    print(f"  k={_k}  docs={[t['doc'] for t in _tr]}  "
          f"tem o D-05: {'SIM' if acertou_em_k(_tr, 'Q-04') else 'NAO'}   "
          f"contexto ~{_chars // 4} tokens")

print()
print("### 3. POR QUE k=1 FALHOU ali, e e a parte que se leva para casa")
for _t in buscar_lexical(PERGUNTAS[3]["texto"], k=3):
    _marca = ""
    if _t["doc"] == "D-01":
        _marca = "<- MAIS parecido, e DESATUALIZADO"
    if _t["doc"] == "D-05":
        _marca = "<- MENOS parecido, e e o que vale"
    print(f"  {_t['score']:.3f}  {_t['doc']}  {_t['titulo'][:38]:<39}{_marca}")
print()
print("  O manual responde '4 mm2' com toda a confianca. O boletim de marco")
print("  de 2026 substituiu esse trecho. E o manual e MAIS parecido com a")
print("  pergunta, porque fala de instalacao o tempo todo.")
print("  Semelhanca nao sabe o que e recente. Isso NAO se resolve com modelo")
print("  melhor: resolve-se com METADADO -- data e situacao do documento --")
print("  filtrando ou reordenando antes de montar o prompt.")

### 1. O TAMANHO DO PEDACO muda quem e recuperado (Q-03)
  tam= 120  melhor=D-07#3   contem a frase decisiva: NAO
  tam= 400  melhor=D-07#1   contem a frase decisiva: SIM
  tam=1200  melhor=D-07#0   contem a frase decisiva: SIM

### 2. O k decide se a armadilha de versao fecha (Q-04)
    pergunta: Que bitola de cabo devo usar no aterramento do VX-200?
  k=1  docs=['D-01']  tem o D-05: NAO   contexto ~91 tokens
  k=3  docs=['D-01', 'D-05', 'D-06']  tem o D-05: SIM   contexto ~269 tokens
  k=5  docs=['D-01', 'D-05', 'D-06', 'D-08', 'D-07']  tem o D-05: SIM   contexto ~410 tokens

### 3. POR QUE k=1 FALHOU ali, e e a parte que se leva para casa
  0.270  D-01  Manual VX-200, secao 3: instalacao     <- MAIS parecido, e DESATUALIZADO
  0.180  D-05  Boletim de servico BS-2026-03          <- MENOS parecido, e e o que vale
  0.128  D-06  Especificacoes tecnicas do VX-200      

  O manual responde '4 mm2' com toda a confianca. O boletim de marco
  de 2026 substituiu esse trecho. E o manual e MA

## Parte 6 — Quem decide ir buscar

Mesma base, mesma instrução. A diferença é a do Encontro 5: na Parte 4 **o
Python decidiu** buscar, e com quais palavras. Agora a busca é uma ferramenta,
e quem decide é o modelo — inclusive decidir **não** buscar, que foi o que
aconteceu no exemplar do Encontro 7.

In [20]:
# ---------------------------------------------------------------------
# RAG COMO FERRAMENTA DO AGENTE -- o fecho com o Encontro 7.
# No exemplar da mesa de servico a base de conhecimento ESTAVA declarada
# como ferramenta, e o agente NAO a chamou em nenhum dos cinco chamados.
# A limitacao 3 do docs/limitacoes.md registra isso. Aqui a mesma
# pergunta fica em pe: ele vai buscar?
# ---------------------------------------------------------------------
# REDE

FERRAMENTAS = [{"type": "function", "function": {
    "name": "buscar_na_base",
    "description": ("Busca trechos da documentacao tecnica da Vertice e devolve "
                    "os mais parecidos com a consulta, cada um com o "
                    "identificador do documento de origem. Cobre manual, "
                    "politicas, boletins de servico, FAQ e glossario."),
    "parameters": {"type": "object", "properties": {
        "consulta": {"type": "string",
                     "description": "o que procurar, em palavras do usuario"}},
        "required": ["consulta"]}}}]


def agente_rag(pergunta: str, max_voltas: int = 5) -> dict:
    msgs = [{"role": "system", "content": INSTRUCAO_RAG + "\n\nUse a ferramenta "
             "disponivel para obter os trechos antes de responder."},
            {"role": "user", "content": pergunta}]
    consultas, entrada, saida, resposta = [], 0, 0, ""
    for volta in range(1, max_voltas + 1):
        r = chamar(msgs, tools=FERRAMENTAS)
        m = r.choices[0].message
        entrada += r.usage.prompt_tokens
        saida += r.usage.completion_tokens
        if not m.tool_calls:
            resposta = m.content or ""
            break
        msgs.append({"role": "assistant", "content": m.content,
                     "tool_calls": [{"id": tc.id, "type": "function",
                                     "function": {"name": tc.function.name,
                                                  "arguments": tc.function.arguments}}
                                    for tc in m.tool_calls]})
        for tc in m.tool_calls:
            args = json.loads(tc.function.arguments or "{}")
            consulta = args.get("consulta", "")
            consultas.append(consulta)
            achados = buscar_lexical(consulta, k=3)
            print(f"  volta {volta}: buscar_na_base({consulta!r})")
            print(f"     -> {[t['doc'] for t in achados]}")
            msgs.append({"role": "tool", "tool_call_id": tc.id,
                         "content": montar_contexto(achados)})
    return {"resposta": resposta, "consultas": consultas,
            "voltas": volta, "custo": custo_usd(entrada, saida)}


_r = agente_rag(PERGUNTAS[3]["texto"])
print()
print("RESPOSTA:", _r["resposta"].strip()[:300])
print()
print(f"consultas feitas: {len(_r['consultas'])} | voltas: {_r['voltas']} | "
      f"US$ {_r['custo']:.6f}")
print()
print("A diferenca com a celula anterior e a mesma do Encontro 5, e vale")
print("dizer em voz alta: ali o Python decidiu buscar e com que palavras.")
print("Aqui quem decide e o modelo -- inclusive decidir NAO buscar.")

  volta 1: buscar_na_base('bitola de cabo aterramento VX-200')
     -> ['D-01', 'D-05', 'D-06']

RESPOSTA: Para o aterramento do chassi do VX‑200, use cabo de 4 mm² ligado ao barramento de terra do painel. [D-01]

consultas feitas: 1 | voltas: 2 | US$ 0.000157

A diferenca com a celula anterior e a mesma do Encontro 5, e vale
dizer em voz alta: ali o Python decidiu buscar e com que palavras.
Aqui quem decide e o modelo -- inclusive decidir NAO buscar.


## Parte 7 — Quando **não** usar RAG

A pergunta que fecha a aula, e ela é a mesma do Encontro 7 em outra roupa:
*o procedimento mais simples já basta?*

In [21]:
# ---------------------------------------------------------------------
# QUANDO NAO USAR RAG. Sem rede.
# ---------------------------------------------------------------------
_tudo = sum(len(d["texto"]) for d in DOCS.values())
_trecho = len(montar_contexto(buscar_lexical(PERGUNTAS[0]["texto"], k=3)))

print(f"corpus inteiro : {_tudo:>6} caracteres  (~{_tudo // 4} tokens)")
print(f"tres trechos   : {_trecho:>6} caracteres  (~{_trecho // 4} tokens)")
print()
print(f"O corpus inteiro cabe no prompt, e custaria cerca de "
      f"US$ {custo_usd(_tudo // 4, 200):.6f} por pergunta.")
print(f"Com recuperacao: cerca de US$ {custo_usd(_trecho // 4, 200):.6f}.")
print()
print("Com OITO documentos, a economia e de centavos de centavo e voce")
print("comprou quatro decisoes novas para errar. RAG passa a valer quando:")
print("  - o corpus nao cabe no prompt, ou cabe mas custa caro por pergunta;")
print("  - o corpus MUDA, e reescrever o prompt a cada mudanca nao escala;")
print("  - a resposta precisa dizer DE ONDE veio, e isso e auditoria.")

corpus inteiro :   2916 caracteres  (~729 tokens)
tres trechos   :   1116 caracteres  (~279 tokens)

O corpus inteiro cabe no prompt, e custaria cerca de US$ 0.000115 por pergunta.
Com recuperacao: cerca de US$ 0.000081.

Com OITO documentos, a economia e de centavos de centavo e voce
comprou quatro decisoes novas para errar. RAG passa a valer quando:
  - o corpus nao cabe no prompt, ou cabe mas custa caro por pergunta;
  - o corpus MUDA, e reescrever o prompt a cada mudanca nao escala;
  - a resposta precisa dizer DE ONDE veio, e isso e auditoria.


## Parte 8 — O que a sua equipe faz com isto

1. **Liste de 5 a 10 documentos** que o seu problema consultaria. Se não
   existirem, o seu problema talvez não peça RAG, e isso é resultado.
2. **Escreva 5 perguntas com verdade de terra em duas colunas:** qual
   documento responde, e o que a resposta certa contém.
3. **Garanta que uma delas contradiga o senso comum.** Sem ela você não
   consegue provar que o RAG está dirigindo a resposta.
4. **Meça a recuperação antes de gastar um token.** `acerto@k` com o
   recuperador por palavra é meia hora de trabalho e já diz se o corpus serve.
5. Só então gere, e classifique cada caso **em um dos quatro quadrantes**.

> O entregável do Marco 2 não é "temos RAG". É a tabela dos quatro
> quadrantes com o seu corpus, e a frase que ela sustenta.

# Referência · o mesmo RAG em LangChain

*Cartão de consulta — não apresentado em aula.*

A biblioteca existe e faz isto em poucas linhas:

```python
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

pedacos = RecursiveCharacterTextSplitter(chunk_size=400,
                                         chunk_overlap=80).split_documents(docs)
indice = FAISS.from_documents(pedacos, GoogleGenerativeAIEmbeddings(
                              model="models/gemini-embedding-001"))
resposta = (indice.as_retriever(search_kwargs={"k": 3}) | prompt | llm).invoke(q)
```

**Cinco linhas no lugar de quarenta, e é exatamente esse o problema hoje.** As
quarenta linhas desta aula existem para deixar visíveis as quatro decisões; as
cinco acima escondem as quatro atrás de valores padrão. Depois de medir à mão,
a biblioteca vira conveniência — que é o que ela deve ser.

**Nota de escopo. O LangChain está aqui como referência de mercado, não como ferramenta adotada. E `FAISS` acima troca a nossa matriz `numpy` por um índice que aguenta milhões de vetores: com 25 pedaços, `numpy` é mais rápido.

# Referência · glossário de RAG

*Cartão de consulta — não apresentado em aula.*

| Termo | O que é, em uma linha |
|---|---|
| *embedding* | vetor de números que representa um texto, de modo que textos parecidos fiquem próximos |
| *chunk* / pedaço | fatia do documento, que é a unidade recuperada |
| *top-k* | quantos pedaços entram no *prompt* |
| cosseno | medida de proximidade entre dois vetores, de -1 a 1 |
| TF-IDF | peso de palavra por frequência; busca léxica clássica |
| busca híbrida | somar o escore léxico ao semântico, porque cada um ganha num caso |
| *re-ranking* | reordenar os k recuperados com um modelo mais caro antes de responder |
| RAG corretivo | o agente avalia o que recuperou e **busca de novo** se não serviu |
| `acerto@k` | o documento certo apareceu entre os k primeiros? Mede **recuperação**, não resposta |

Os dois últimos são a extensão natural desta aula e estão na lição 09 do curso
da Microsoft.